# 1. Project Overview

The UCI SECOM (Semiconductor Manufacturing) dataset is a classic, highly imbalanced benchmark dataset used for binary classification and feature selection. It represents a real-world manufacturing scenario where signals from hundreds of sensors are used to predict whether a semiconductor wafer will pass or fail a quality test.


## Project Goal
To build optimised models that will efficiently predict whether a semiconductor wafer will pass or fail a quality test.


## Dataset Overview
- **Dataset Name:** UCI SECOM (Semiconductor Manufacturing) dataset
- **Source:** [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/179/secom) 
- **Total Instances:** 1,567 wafers.
- **Total Features:** 591 attributes, primarily representing sensor measurements and process points.
- **Target Variable:** Binary classification (Pass/Fail).
Fail (+1): 104 instances (approx. 6.6%).Pass (-1): 1,463 instances.

## Core Challenges

- **Extreme Class Imbalance:** With only about 6.6% "fail" cases, traditional models often achieve high accuracy by simply predicting "pass" every time, making recall more important than overall accuracy.
- **High Dimensionality:** Many of the 591 features are redundant, contain noise, or have zero variance (the same value for every entry), requiring robust feature selection.
- **Missing Data:** The dataset contains a significant number of null/NaN values, varying in density across different features, which necessitates careful imputation



- **Notes** The Dataset includes .

# Problem Solving

## Import Required Libraries and Dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

In [3]:
from pathlib import Path
import pandas as pd

project_root = Path('..') 

raw_data_path = project_root / 'data' / 'raw'
processed_data_path = project_root / 'data' / 'processed'

data = pd.read_csv(raw_data_path / 'uci-secom.csv')
data.head(100)

,Time,0,1,2,3,4,5,6,7,8,...,581,582,583,584,585,586,587,588,589,Pass/Fail
0,2008-07-19 11:55:00,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1
1,2008-07-19 12:32:00,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2008-07-19 13:17:00,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2008-07-19 14:43:00,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,2008-07-19 15:22:00,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2008-04-08 20:32:00,3081.07,2560.99,2180.1556,1822.5073,1.2579,100.0,98.1289,0.1261,1.5048,...,NaN,0.5002,0.0098,0.0027,1.9607,0.0277,0.0318,0.0097,114.7497,-1
96,2008-04-08 20:58:00,2992.40,2467.07,2191.6667,1107.4330,1.3529,100.0,103.4233,0.1206,1.4993,...,192.2985,0.4996,0.0326,0.0065,6.5274,0.0095,0.0184,0.0062,192.2985,1
97,2008-04-08 21:43:00,3049.31,2453.82,2265.1889,1740.3297,1.4715,100.0,100.3889,0.1222,1.5310,...,NaN,0.4998,0.0174,0.0049,3.4900,0.0095,0.0184,0.0062,192.2985,-1
98,2008-04-08 22:51:00,3011.49,2537.90,2183.4556,955.9073,1.1048,100.0,102.6978,0.1223,1.6227,...,NaN,0.4955,0.0110,0.0032,2.2104,0.0095,0.0184,0.0062,192.2985,-1


## Exploratory Data Analysis


Examining the Data Structure

In [75]:
df = data.copy()
display(df.head(5))


,Time,0,1,2,3,4,5,6,7,8,...,581,582,583,584,585,586,587,588,589,Pass/Fail
0,2008-07-19 11:55:00,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1
1,2008-07-19 12:32:00,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2008-07-19 13:17:00,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2008-07-19 14:43:00,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,2008-07-19 15:22:00,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1


In [76]:
display(df.info())
display(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1567 entries, 0 to 1566
Columns: 592 entries, Time to Pass/Fail
dtypes: float64(590), int64(1), object(1)
memory usage: 7.1+ MB


None

,0,1,2,3,4,5,6,7,8,9,...,581,582,583,584,585,586,587,588,589,Pass/Fail
count,1561.000000,1560.000000,1553.000000,1553.000000,1553.000000,1553.0,1553.000000,1558.000000,1565.000000,1565.000000,...,618.000000,1566.000000,1566.000000,1566.000000,1566.000000,1566.000000,1566.000000,1566.000000,1566.000000,1567.000000
mean,3014.452896,2495.850231,2200.547318,1396.376627,4.197013,100.0,101.112908,0.121822,1.462862,-0.000841,...,97.934373,0.500096,0.015318,0.003847,3.067826,0.021458,0.016475,0.005283,99.670066,-0.867262
std,73.621787,80.407705,29.513152,441.691640,56.355540,0.0,6.237214,0.008961,0.073897,0.015116,...,87.520966,0.003404,0.017180,0.003720,3.578033,0.012358,0.008808,0.002867,93.891919,0.498010
min,2743.240000,2158.750000,2060.660000,0.000000,0.681500,100.0,82.131100,0.000000,1.191000,-0.053400,...,0.000000,0.477800,0.006000,0.001700,1.197500,-0.016900,0.003200,0.001000,0.000000,-1.000000
25%,2966.260000,2452.247500,2181.044400,1081.875800,1.017700,100.0,97.920000,0.121100,1.411200,-0.010800,...,46.184900,0.497900,0.011600,0.003100,2.306500,0.013425,0.010600,0.003300,44.368600,-1.000000
50%,3011.490000,2499.405000,2201.066700,1285.214400,1.316800,100.0,101.512200,0.122400,1.461600,-0.001300,...,72.288900,0.500200,0.013800,0.003600,2.757650,0.020500,0.014800,0.004600,71.900500,-1.000000
75%,3056.650000,2538.822500,2218.055500,1591.223500,1.525700,100.0,104.586700,0.123800,1.516900,0.008400,...,116.539150,0.502375,0.016500,0.004100,3.295175,0.027600,0.020300,0.006400,114.749700,-1.000000
max,3356.350000,2846.440000,2315.266700,3715.041700,1114.536600,100.0,129.252200,0.128600,1.656400,0.074900,...,737.304800,0.509800,0.476600,0.104500,99.303200,0.102800,0.079900,0.028600,737.304800,1.000000


In [77]:
print(df.shape, '\n(rows and columns)')

(1567, 592) 
(rows and columns)


In [78]:
display(df.dtypes.value_counts())

float64    590
object       1
int64        1
Name: count, dtype: int64

In [79]:
df.select_dtypes(include=['int64', 'object']).head()

,Time,Pass/Fail
0,2008-07-19 11:55:00,-1
1,2008-07-19 12:32:00,-1
2,2008-07-19 13:17:00,1
3,2008-07-19 14:43:00,-1
4,2008-07-19 15:22:00,-1


In [80]:
# checking for duplicates
df[df.duplicated()]

,Time,0,1,2,3,4,5,6,7,8,...,581,582,583,584,585,586,587,588,589,Pass/Fail


In [81]:
# checking for null values
null_list=df.isna().sum().to_list()
print(null_list)

[0, 6, 7, 14, 14, 14, 14, 14, 9, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 10, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 24, 24, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 7, 6, 6, 6, 7, 7, 7, 6, 6, 6, 6, 6, 6, 794, 794, 6, 24, 24, 24, 24, 24, 24, 24, 24, 1, 12, 1341, 0, 0, 0, 51, 51, 6, 2, 2, 6, 6, 6, 6, 6, 6, 6, 6, 6, 2, 2, 6, 6, 6, 6, 1018, 1018, 1018, 715, 0, 0, 0, 0, 0, 24, 0, 0, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 5, 6, 7, 14, 14, 14, 14, 14, 9, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 10, 0, 1429, 1429, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 24, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 7, 6, 6, 6, 7, 7, 7, 6, 6, 6, 6, 6, 6, 6, 24, 24, 24, 24, 24, 24, 24, 24, 1, 12, 1341, 0, 0, 0, 51, 51, 6, 2, 2, 6, 6, 6, 6, 6, 6, 6, 6, 6, 2, 2, 6, 6, 6, 6, 1018, 1018, 1018, 715, 0, 0, 0, 0, 0, 24, 0, 0, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 5, 6, 7, 14, 14, 14, 14, 14, 9, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 10, 0, 1429, 1429, 2, 2, 2, 2, 2, 2, 2, 2, 2

#### Note:

From the above Data Exploration, it can be deduced that our data has 1567 rows and 592 columns.
The columns are in their accurate datatypes and there are no duplicates.

Also, it is observed that there is a lot of varying number of missing values across multiple columns.
Probably due to sensor failures or unrecorded readings.
Some columns have missing values of over 1000 which is about 63% of the data.

Therefore we will need to drop columns that has  more than half of its entries as missing or null as they will only serve as noise to our model.

## Data Pre-processing

In [82]:
# Evaluating the percentages of missing values
missing_pct = ((df.isnull().mean() * 100).round()).sort_values(ascending=False)
missing_pct

293          91.0
158          91.0
157          91.0
292          91.0
85           86.0
             ... 
222           0.0
221           0.0
218           0.0
209           0.0
Pass/Fail     0.0
Length: 592, dtype: float64

In [83]:
# Setting our threshold 
threshold = 50  # 50%
cols_to_drop = missing_pct[missing_pct > threshold].index
df = df.drop(columns=cols_to_drop)
print('Total No. of Columns dropped: ', len(cols_to_drop))

Total No. of Columns dropped:  28


We can now fill the missing values For the Remaining columns with their mean but first we will need to seperate the Non-numerical columns from the Numerical

In [100]:
x = df.drop(columns= ['Time', 'Pass/Fail']) 
y = df['Pass/Fail']

0      -1
1      -1
2       1
3      -1
4      -1
       ..
1562   -1
1563   -1
1564   -1
1565   -1
1566   -1
Name: Pass/Fail, Length: 1567, dtype: int64

In [102]:
# filling the missing values For the Remaining columns with their mean
X = x.fillna(x.mean())
X.isnull().sum().sum() # for validation

0

Next we are to remove near-constant columns.

This are columns thave have little to no variance across all records of the dataset

In [104]:
display(X.shape)
variances = X.var()
low_var_cols = variances[variances < 0.01].index

X = X.drop(columns=low_var_cols)
print('Total No. of columns dropped : ', variances[variances < 0.01].count(), '\n\n New Dataset Shape:', X.shape)

(1567, 297)

Total No. of columns dropped :  0 

 New Dataset Shape: (1567, 297)


Next, we compute the Correlation matrix to ensure multiple features do not carry the same information

We shall Drop highly correlated features to Prevent multicollinearity and Reduce redundancy.

In [105]:
corr_matrix = (X).corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]

X = X.drop(columns=to_drop)
X.shape


(1567, 195)

In [106]:
# Confirming the number of classes
y.value_counts()

Pass/Fail
-1    1463
 1     104
Name: count, dtype: int64

Number of passes is really low at about 7% of the entire dataset.

This confirms that there is a Class Imbalance therfore we must take this into consideration while building the models

## Modelling

In [107]:
# Train-test Split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [108]:
# Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Logistic Regression Model

In [109]:
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [110]:

logreg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
logreg.fit(X_train_scaled, y_train)

y_pred_lr = logreg.predict(X_test_scaled)
y_proba_lr = logreg.predict_proba(X_test_scaled)[:,1]

print("Logistic Regression")
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))
print("ROC AUC:", roc_auc_score(y_test, y_proba_lr))

Logistic Regression
[[232  58]
 [ 13  11]]
              precision    recall  f1-score   support

          -1       0.95      0.80      0.87       290
           1       0.16      0.46      0.24        24

    accuracy                           0.77       314
   macro avg       0.55      0.63      0.55       314
weighted avg       0.89      0.77      0.82       314

ROC AUC: 0.662787356321839


### Random Forest Classifier Model 

In [111]:
rf = RandomForestClassifier(class_weight='balanced', n_estimators=200, random_state=42)
rf.fit(X_train, y_train) # We do not fit tree based models to scaled data

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:,1]

print("Random Forest")
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))
print("ROC AUC:", roc_auc_score(y_test, y_proba_rf)) 

Random Forest
[[290   0]
 [ 24   0]]
              precision    recall  f1-score   support

          -1       0.92      1.00      0.96       290
           1       0.00      0.00      0.00        24

    accuracy                           0.92       314
   macro avg       0.46      0.50      0.48       314
weighted avg       0.85      0.92      0.89       314

ROC AUC: 0.7288074712643678


C:\Users\CJ\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\CJ\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\CJ\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


This warning pops up because your model is completely ignoring one or more classes in its predictions.When a model predicts 0 samples for a specific label, the formula for Precision involves dividing by zero. Scikit-learn defaults that result to 0.0 and throws this warning.Why is this happening?Imbalanced Data: Your dataset might have very few examples of certain classes, so the model finds it "easier" to just guess the majority class every time.Weak Model: The model hasn't learned enough patterns to confidently pick the minority labels.Small Test Set: If your test set is tiny, it's possible the model just missed those few instances by chance.How to fix it1. Quick fix: Silence the warningIf you just want the code to run without the wall of text, add the zero_division parameter to your classification report or metric:pythonfrom sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, zero_division=0))
Use code with caution.2. Real fix: Address the data imbalanceIf you want the model to actually perform better, try these:Stratified Split: Ensure your train/test split has the same percentage of each class.pythonX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
Use code with caution.Class Weights: Most Scikit-learn models (like RandomForest or SVC) have a parameter to handle imbalance automatically.pythonmodel = RandomForestClassifier(class_weight='balanced')
Use code with caution.Oversampling: Use a tool like SMOTE to create synthetic examples of the rare classes.Do you have a huge difference in the number of samples between your majority and minority classes?

### Support Vector Classifier Model

In [112]:


svc_model = SVC(class_weight='balanced', max_iter=1000, random_state=42)
svc_model.fit(X_train_scaled, y_train)
y_pred_svc = rf.predict(X_test_scaled)
y_proba_svc = rf.predict_proba(X_test_scaled)[:,1]

print("SVC")
print(confusion_matrix(y_test, y_pred_svc))
print(classification_report(y_test, y_pred_svc))
print("ROC AUC:", roc_auc_score(y_test, y_proba_svc))

SVC
[[290   0]
 [ 24   0]]
              precision    recall  f1-score   support

          -1       0.92      1.00      0.96       290
           1       0.00      0.00      0.00        24

    accuracy                           0.92       314
   macro avg       0.46      0.50      0.48       314
weighted avg       0.85      0.92      0.89       314

ROC AUC: 0.5663074712643678


C:\Users\CJ\anaconda3\Lib\site-packages\sklearn\svm\_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
C:\Users\CJ\anaconda3\Lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\CJ\anaconda3\Lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\CJ\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\CJ\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarnin

## Model Evaluation

Comparing the Models

In [115]:
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'SVC'],
    
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_svc),],
    'Recall (Failure)': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_svc),],
    'ROC AUC': [
        roc_auc_score(y_test, y_proba_lr),
        roc_auc_score(y_test, y_proba_rf),
        roc_auc_score(y_test, y_proba_svc),]
})

results.sort_values('Recall (Failure)', ascending=False)


,Model,Accuracy,Recall (Failure),ROC AUC
0,Logistic Regression,0.773885,0.458333,0.662787
1,Random Forest,0.923567,0.000000,0.728807
2,SVC,0.923567,0.000000,0.566307


DataPoints with Test/Fail Records = 1 are the defective wafers and they are rare and very critical as the cost of missing a failure is significantly higher than raising a false alarm. 
Due to the great disparity in class distribution metrics such as Accuracy score can be misleading 

For this reason we must focus our evaluation on the RECALL metric which will tell us how many failures we were able to identify out of all the failures recorded. 

A low recall value means many failures slipped through undetected. 
Therefore detecting as many failure cases as possible is critical to preventing downstream risks




The RandomForest and Support Vector Classifier models both have high accuracy scores (> 90%) but couldnt detect the failures 

while the logistic regression model though having a lower accuracy score (77%) was able to detect some failures.

*IN SUMMARY:*

In predicting whether a semiconductor wafer will pass or fail the quality test the three trained models performed poorly.

The logistic Regression model has the best performance with the ability to detect some failures

**RECOMMENDATION/WAY FORWARD**

To try other sampling methods to even the effects of class imbalance

Identify the most important features

To optimise the models using hyperparameter tuning


